In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd  
import numpy as np

### Stage 1: Data Ingestion

In [4]:
deliveries_df=pd.read_csv("dataset/deliveries.csv")
matches_df=pd.read_csv("dataset/matches.csv")
print(deliveries_df.shape)
print(matches_df.shape)
deliveries_df.head()
matches_df.head()
deliveries_df.info()
matches_df.info()


(260920, 17)
(1095, 20)
<class 'pandas.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   match_id          260920 non-null  int64
 1   inning            260920 non-null  int64
 2   batting_team      260920 non-null  str  
 3   bowling_team      260920 non-null  str  
 4   over              260920 non-null  int64
 5   ball              260920 non-null  int64
 6   batter            260920 non-null  str  
 7   bowler            260920 non-null  str  
 8   non_striker       260920 non-null  str  
 9   batsman_runs      260920 non-null  int64
 10  extra_runs        260920 non-null  int64
 11  total_runs        260920 non-null  int64
 12  extras_type       14125 non-null   str  
 13  is_wicket         260920 non-null  int64
 14  player_dismissed  12950 non-null   str  
 15  dismissal_kind    12950 non-null   str  
 16  fielder           9354 non-null    str  
dt

### Stage 2: Data Cleaning & Validation


In [5]:
deliveries_df.isnull().sum().sort_values(ascending=False)


fielder             251566
dismissal_kind      247970
player_dismissed    247970
extras_type         246795
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
non_striker              0
bowler                   0
batter                   0
ball                     0
total_runs               0
extra_runs               0
batsman_runs             0
is_wicket                0
dtype: int64

In [6]:
matches_df.isnull().sum().sort_values(ascending=False)

method             1074
city                 51
result_margin        19
player_of_match       5
winner                5
target_runs           3
target_overs          3
id                    0
date                  0
season                0
venue                 0
match_type            0
toss_decision         0
toss_winner           0
team2                 0
team1                 0
result                0
super_over            0
umpire1               0
umpire2               0
dtype: int64

In [7]:
deliveries_df['dismissal_kind'] = deliveries_df['dismissal_kind'].fillna('None')
deliveries_df['player_dismissed'] = deliveries_df['player_dismissed'].fillna('None')
deliveries_df['fielder'] = deliveries_df['fielder'].fillna('Not Applicable')
deliveries_df['extras_type'] = deliveries_df['extras_type'].fillna('None')

In [8]:
deliveries_df.isnull().sum()

match_id            0
inning              0
batting_team        0
bowling_team        0
over                0
ball                0
batter              0
bowler              0
non_striker         0
batsman_runs        0
extra_runs          0
total_runs          0
extras_type         0
is_wicket           0
player_dismissed    0
dismissal_kind      0
fielder             0
dtype: int64

In [9]:
invalid_rows = deliveries_df[
    (deliveries_df['is_wicket'] == 0) &
    (deliveries_df['player_dismissed'] != 'None')
]

print(len(invalid_rows))

0


In [10]:
print(deliveries_df.isnull().sum().sum())
print(matches_df.isnull().sum().sum())

set(deliveries_df['match_id']) - set(matches_df['id'])

print(deliveries_df.duplicated().sum())
print(matches_df.duplicated().sum())

0
1160
0
0


In [11]:
matches_df['method'] = matches_df['method'].fillna('Normal')
matches_df['city'] = matches_df['city'].fillna('Unknown')
matches_df['winner'] = matches_df['winner'].fillna('No Result')
matches_df['player_of_match'] = matches_df['player_of_match'].fillna('Not Awarded')
matches_df['result_margin'] = matches_df['result_margin'].fillna(-1)
matches_df['target_runs'] = matches_df['target_runs'].fillna(0)
matches_df['target_overs'] = matches_df['target_overs'].fillna(0)

In [12]:
deliveries_df['dismissal_kind'] = deliveries_df['dismissal_kind'].fillna('None')
deliveries_df['player_dismissed'] = deliveries_df['player_dismissed'].fillna('None')
deliveries_df['fielder'] = deliveries_df['fielder'].fillna('Not Applicable')
deliveries_df['extras_type'] = deliveries_df['extras_type'].fillna('None')

In [13]:
delivery_ids = set(deliveries_df['match_id'])
match_ids = set(matches_df['id'])
missing_in_matches = delivery_ids - match_ids
missing_in_deliveries = match_ids - delivery_ids
print("Missing in matches:", len(missing_in_matches))
print("Missing in deliveries:", len(missing_in_deliveries))
valid_ids = delivery_ids.intersection(match_ids)
deliveries_df = deliveries_df[deliveries_df['match_id'].isin(valid_ids)]
matches_df = matches_df[matches_df['id'].isin(valid_ids)]


Missing in matches: 0
Missing in deliveries: 0


In [14]:
# matches_df['season'] = matches_df['season'].astype(int)

# Deliveries numeric columns
deliveries_df['batsman_runs'] = deliveries_df['batsman_runs'].astype(int)
deliveries_df['extra_runs'] = deliveries_df['extra_runs'].astype(int)
deliveries_df['total_runs'] = deliveries_df['total_runs'].astype(int)
deliveries_df['over'] = deliveries_df['over'].astype(int)
deliveries_df['ball'] = deliveries_df['ball'].astype(int)

In [15]:
matches_df.isnull().sum().sort_values(ascending=False)

id                 0
season             0
city               0
date               0
match_type         0
player_of_match    0
venue              0
team1              0
team2              0
toss_winner        0
toss_decision      0
winner             0
result             0
result_margin      0
target_runs        0
target_overs       0
super_over         0
method             0
umpire1            0
umpire2            0
dtype: int64

In [16]:
matches_df['date'] = pd.to_datetime(matches_df['date'])

### Stage 3: Data Transformation

In [17]:
(deliveries_df['total_runs'] == 
 (deliveries_df['batsman_runs'] + deliveries_df['extra_runs'])).all()

np.True_

In [18]:
matches_df.rename(columns={'id': 'match_id'}, inplace=True)


In [19]:
ipl_df = deliveries_df.merge(
    matches_df,
    on='match_id',
    how='inner'
)

In [20]:
print("Shape of merged dataset:", ipl_df.shape)
ipl_df.head()

Shape of merged dataset: (260920, 36)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,Normal,Asad Rauf,RE Koertzen
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,Normal,Asad Rauf,RE Koertzen
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,Normal,Asad Rauf,RE Koertzen
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,Normal,Asad Rauf,RE Koertzen
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,Normal,Asad Rauf,RE Koertzen


### Stage 4: Core Analysis

1. Total Runs per Match

In [21]:
runs_per_match = (
    ipl_df.groupby('match_id')['total_runs']
    .sum()
    .reset_index(name='total_runs')
)

runs_per_match.head()

,match_id,total_runs
0,335982,304
1,335983,447
2,335984,261
3,335985,331
4,335986,222


2. Runs per Team per Match

In [22]:
runs_per_team_match = (
    ipl_df.groupby(['match_id', 'batting_team'])['total_runs']
    .sum()
    .reset_index(name='team_runs')
)

runs_per_team_match.head()

,match_id,batting_team,team_runs
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Kings XI Punjab,207
4,335984,Delhi Daredevils,132


3. Top 10 Batters

In [23]:
top_batters = (
    ipl_df.groupby('batter')['batsman_runs']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='total_runs')
)

top_batters

,batter,total_runs
0,V Kohli,8014
1,S Dhawan,6769
2,RG Sharma,6630
3,DA Warner,6567
4,SK Raina,5536
5,MS Dhoni,5243
6,AB de Villiers,5181
7,CH Gayle,4997
8,RV Uthappa,4954
9,KD Karthik,4843


4. Strike Rate of Batters

In [24]:
batter_stats = (
    ipl_df.groupby('batter')
    .agg(
        runs=('batsman_runs', 'sum'),
        balls=('batsman_runs', 'count')
    )
    .reset_index()
)
batter_stats['strike_rate'] = (batter_stats['runs'] / batter_stats['balls']) * 100
batter_stats.head()

,batter,runs,balls,strike_rate
0,A Ashish Reddy,280,196,142.857143
1,A Badoni,634,505,125.544554
2,A Chandila,4,7,57.142857
3,A Chopra,53,75,70.666667
4,A Choudhary,25,20,125.000000


5. Top 10 Bowlers by Economy

In [25]:
bowler_stats=(
    ipl_df.groupby('bowler')
    .agg(
        runs_conceded=('total_runs','sum'),
        bowled_balls=('total_runs','count')
    )
    .reset_index()
)
bowler_stats['overs']=bowler_stats['bowled_balls']/6;
bowler_stats['economy']=bowler_stats['runs_conceded']/bowler_stats['overs']
bowler_stats=bowler_stats[bowler_stats['overs']>200]
top_economy_bowlers=(bowler_stats.sort_values('economy').head(10))
top_economy_bowlers

,bowler,runs_conceded,bowled_balls,overs,economy
263,M Muralitharan,1765,1581,263.500000,6.698292
446,SP Narine,4672,4146,691.000000,6.761216
138,DW Steyn,2583,2282,380.333333,6.791411
398,Rashid Khan,3340,2901,483.500000,6.907963
355,R Ashwin,5435,4679,779.833333,6.969438
438,SL Malinga,3486,2974,495.666667,7.032952
163,Harbhajan Singh,4101,3496,582.666667,7.038330
437,SK Warne,1465,1223,203.833333,7.187244
193,JJ Bumrah,3840,3185,530.833333,7.233909
221,KH Pandya,2644,2183,363.833333,7.267064


6. Most Consistent Batters

In [26]:
batter_match_runs = (
    ipl_df.groupby(['batter', 'match_id'])['batsman_runs']
    .sum()
    .reset_index()
)
consistent_batters = (
    batter_match_runs.groupby('batter')
    .agg(
        total_runs=('batsman_runs', 'sum'),
        matches=('match_id', 'nunique'),
        avg_runs=('batsman_runs', 'mean')
    )
    .reset_index()
)
consistent_batters = consistent_batters[consistent_batters['matches'] >= 30]
consistent_batters = consistent_batters.sort_values('avg_runs', ascending=False)
consistent_batters.head(10)

,batter,total_runs,matches,avg_runs
289,KL Rahul,4689,122,38.434426
473,RD Gaikwad,2380,65,36.615385
542,SE Marsh,2489,69,36.072464
147,DA Warner,6567,184,35.690217
124,CH Gayle,4997,141,35.439716
365,ML Hayden,1107,32,34.593750
352,MEK Hussey,1977,58,34.086207
242,JC Buttler,3583,106,33.801887
188,F du Plessis,4571,138,33.123188
631,V Kohli,8014,244,32.844262


7. Highest Individual Score in a Match

In [27]:
batsman_runs_per_match=(
    ipl_df.groupby(['batter','match_id'])['batsman_runs']
    .sum()
    .reset_index()
)
highest_score=batsman_runs_per_match.sort_values('batsman_runs',ascending=False)
highest_score.head(1)

,batter,match_id,batsman_runs
2469,CH Gayle,598027,175


8. Boundary Analysis

In [28]:
no_of_fours=(ipl_df['batsman_runs']==4).sum()
no_of_sixers=(ipl_df['batsman_runs']==6).sum()
print(no_of_fours)
print(no_of_sixers)

29850
13051


In [29]:
boundary_df=ipl_df[ipl_df['batsman_runs'].isin([4,6])]
top_boundary_batter=(boundary_df.groupby('batter')['batsman_runs'].count()
                    .sort_values(ascending=False)
                    .head(10)
                    .reset_index(name='boundary_count'))
top_boundary_batter

,batter,boundary_count
0,V Kohli,981
1,S Dhawan,921
2,DA Warner,899
3,RG Sharma,880
4,CH Gayle,767
5,SK Raina,710
6,AB de Villiers,667
7,RV Uthappa,663
8,KD Karthik,627
9,MS Dhoni,615


9. Boundary Percentage

In [30]:
total_runs=(ipl_df.groupby('batter')['batsman_runs'].sum().reset_index(name='total_runs'))
boundary_runs=(ipl_df[ipl_df['batsman_runs'].isin([4,6])].groupby('batter')['batsman_runs'].sum().reset_index(name='boundary_runs'))
boundary_percentage = total_runs.merge(boundary_runs,on='batter',how='left')
boundary_percentage['boundary_runs'] = boundary_percentage['boundary_runs'].fillna(0)
boundary_percentage['boundary_percentage'] = (boundary_percentage['boundary_runs'] /boundary_percentage['total_runs']) * 100
boundary_percentage.head()

,batter,total_runs,boundary_runs,boundary_percentage
0,A Ashish Reddy,280,154.0,55.000000
1,A Badoni,634,328.0,51.735016
2,A Chandila,4,0.0,0.000000
3,A Chopra,53,28.0,52.830189
4,A Choudhary,25,10.0,40.000000


10. Dot Ball Analysis

In [31]:
dot_balls = (ipl_df['total_runs'] == 0).sum()
print("Total dot balls:", dot_balls)

Total dot balls: 90438


In [32]:
dot_ball_df = ipl_df[ipl_df['total_runs'] == 0]

top_dot_bowlers = (
    dot_ball_df.groupby('bowler')
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='dot_balls')
)
top_dot_bowlers

,bowler,dot_balls
0,B Kumar,1632
1,SP Narine,1569
2,R Ashwin,1552
3,PP Chawla,1325
4,Harbhajan Singh,1263
5,JJ Bumrah,1228
6,RA Jadeja,1216
7,YS Chahal,1194
8,UT Yadav,1186
9,A Mishra,1185


11. Runs per Over Analysis

In [ ]:
ipl_df = ipl_df[ipl_df['over'].between(1, 2)]
runs_per_over = (
    ipl_df.groupby('over')['total_runs']
    .mean()
    .reset_index(name='avg_runs')
)
runs_per_over['runs_per_over'] = runs_per_over['avg_runs'] * 6
runs_per_over = runs_per_over[['over', 'runs_per_over']]
runs_per_over

,over,runs_per_over
0,1,7.041168
1,2,7.896595
2,3,8.139227
3,4,8.214602
4,5,8.238328
5,6,6.618644
6,7,7.139687
7,8,7.452523
8,9,7.347461
9,10,7.575739


In [40]:
high_scoring_overs = runs_per_over.sort_values('runs_per_over', ascending=False).head(5)
high_scoring_overs

,over,runs_per_over
18,19,10.661132
17,18,9.881378
16,17,9.527034
15,16,8.992669
14,15,8.605637


12. Powerplay Performance (Overs 1–6)

In [42]:
powerplay_df = ipl_df[ipl_df['over'] <= 6]
powerplay_total = powerplay_df['total_runs'].sum()
powerplay_total

np.int64(104405)

In [ ]:
powerplay_teams = (
    powerplay_df.groupby('batting_team')['total_runs']
    .sum()
    .sort_values(ascending=False)
    .reset_index(name='powerplay_runs')
)
powerplay_teams.head(10)

,batting_team,powerplay_runs
0,Mumbai Indians,12371
1,Kolkata Knight Riders,11912
2,Chennai Super Kings,11379
3,Royal Challengers Bangalore,10929
4,Rajasthan Royals,10333
5,Kings XI Punjab,9095
6,Sunrisers Hyderabad,9047
7,Delhi Daredevils,7506
8,Delhi Capitals,4706
9,Deccan Chargers,3407


13. Death Overs Performance (Overs 16–20)


In [45]:
death_df = ipl_df[ipl_df['over'] >= 16]
death_total = death_df['total_runs'].sum()
death_total

np.int64(75412)

In [46]:
death_teams = (
    death_df.groupby('batting_team')['total_runs']
    .sum()
    .sort_values(ascending=False)
    .reset_index(name='death_runs')
)
death_teams.head(10)

,batting_team,death_runs
0,Mumbai Indians,9598
1,Chennai Super Kings,9061
2,Royal Challengers Bangalore,8417
3,Kolkata Knight Riders,8053
4,Rajasthan Royals,7281
5,Sunrisers Hyderabad,6237
6,Kings XI Punjab,6227
7,Delhi Daredevils,5043
8,Delhi Capitals,3141
9,Deccan Chargers,2539


In [47]:
death_batters = (
    death_df.groupby('batter')['batsman_runs']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='death_runs')
)

death_batters

,batter,death_runs
0,MS Dhoni,2786
1,KA Pollard,1708
2,KD Karthik,1565
3,AB de Villiers,1421
4,RA Jadeja,1420
5,RG Sharma,1176
6,HH Pandya,1126
7,V Kohli,1099
8,AD Russell,1065
9,DA Miller,988


14. Run Distribution per Inning

In [48]:
inning_runs = (
    ipl_df.groupby(['match_id', 'inning'])['total_runs']
    .sum()
    .reset_index()
)
inning_distribution = (
    inning_runs.groupby('inning')['total_runs']
    .mean()
    .reset_index(name='avg_runs')
)
inning_distribution

,inning,avg_runs
0,1,159.679452
1,2,145.838828


15. Toss Impact Analysis

In [ ]:
ipl_df['is_toss_winner_batting'] = (ipl_df['batting_team'] == ipl_df['toss_winner'])

team_runs = (ipl_df.groupby(['match_id', 'batting_team'])['total_runs'].sum().reset_index())
team_runs = team_runs.merge(matches_df[['match_id', 'toss_winner']],on='match_id')
team_runs['is_toss_winner'] = (team_runs['batting_team'] == team_runs['toss_winner'])
toss_analysis = (team_runs.groupby('is_toss_winner')['total_runs'].mean().reset_index(name='avg_runs'))

toss_analysis

,is_toss_winner,avg_runs
0,False,155.304388
1,True,150.230558


16. Player of Match Contribution

In [55]:
batter_match_runs = (ipl_df.groupby(['match_id', 'batter'])['batsman_runs'].sum().reset_index())
top_scorers = (batter_match_runs.sort_values(['match_id', 'batsman_runs'], ascending=[True, False]).drop_duplicates('match_id'))
pom_check = top_scorers.merge(matches_df[['match_id', 'player_of_match']],on='match_id')
pom_check['is_top_scorer'] = (pom_check['batter'] == pom_check['player_of_match'])
pom_accuracy = pom_check['is_top_scorer'].mean()
print(pom_accuracy*100,'%')

44.29223744292237 %


17. Venue-wise Analysis

In [57]:
matches_per_venue = (matches_df.groupby('venue')['match_id'].count().reset_index(name='total_matches'))
venue_runs = (ipl_df.groupby(['match_id', 'venue'])['total_runs'].sum().reset_index())
avg_runs_per_venue = (venue_runs.groupby('venue')['total_runs'].mean().reset_index(name='avg_runs'))
avg_runs_per_venue.sort_values('avg_runs', ascending=False).head()

,venue,avg_runs
12,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,386.000000
24,"M Chinnaswamy Stadium, Bengaluru",366.214286
19,"Himachal Pradesh Cricket Association Stadium, ...",366.000000
1,"Arun Jaitley Stadium, Delhi",364.312500
15,"Eden Gardens, Kolkata",362.250000


18. City-wise Scoring Trends

In [60]:
city_runs = (ipl_df.groupby(['match_id', 'city'])['total_runs'].sum().reset_index())
avg_runs_city = (city_runs.groupby('city')['total_runs'].mean().reset_index(name='avg_runs'))
avg_runs_city.sort_values('avg_runs', ascending=False).head(15)

,city,avg_runs
3,Bengaluru,345.517241
11,Dharamsala,327.923077
15,Guwahati,323.333333
25,Mohali,320.400000
1,Ahmedabad,318.194444
32,Rajkot,316.600000
26,Mumbai,316.231214
7,Chandigarh,313.737705
9,Cuttack,313.000000
20,Kanpur,310.750000


19. Season-wise Run Trends


In [63]:
season_runs = (ipl_df.groupby(['season', 'match_id'])['total_runs'].sum().reset_index())
season_trend = (season_runs.groupby('season')['total_runs'].sum().reset_index(name='total_runs'))
season_trend
season_avg = (season_runs.groupby('season')['total_runs'].mean().reset_index(name='avg_runs_per_match'))
season_avg

,season,avg_runs_per_match
0,2007/08,297.137931
1,2009,275.192982
2,2009/10,301.633333
3,2011,277.150685
4,2012,292.486486
5,2013,286.263158
6,2014,302.250000
7,2015,299.000000
8,2016,302.616667
9,2017,305.050847


20. Winning Team Analysis


In [64]:
# Runs per team per match
team_runs = (
    ipl_df.groupby(['match_id', 'batting_team'])['total_runs']
    .sum()
    .reset_index()
)

# Get highest scoring team
predicted_winner = (
    team_runs.sort_values(['match_id', 'total_runs'], ascending=[True, False])
    .drop_duplicates('match_id')
)

predicted_winner = predicted_winner.rename(
    columns={'batting_team': 'predicted_winner'}
)

In [65]:
winner_comparison = predicted_winner.merge(
    matches_df[['match_id', 'winner']],
    on='match_id'
)

winner_comparison['correct_prediction'] = (
    winner_comparison['predicted_winner'] == winner_comparison['winner']
)

accuracy = winner_comparison['correct_prediction'].mean()

accuracy

np.float64(0.7479452054794521)

### Stage-5:Derived Insights

In [67]:
most_consistent_batter = consistent_batters.head(1)
most_consistent_batter

,batter,total_runs,matches,avg_runs
289,KL Rahul,4689,122,38.434426


- The most consistent batter is KL Rahul, with the highest average runs per match among players with significant match participation.


In [68]:
best_death_team = death_teams.head(1)
best_death_team

,batting_team,death_runs
0,Mumbai Indians,9598


- Mumbai Indians is the strongest team in death overs, scoring the highest total runs in overs 16–20.


In [70]:
high_scoring_venues = avg_runs_per_venue.sort_values('avg_runs', ascending=False).head(5)
high_scoring_venues

,venue,avg_runs
12,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,386.000000
24,"M Chinnaswamy Stadium, Bengaluru",366.214286
19,"Himachal Pradesh Cricket Association Stadium, ...",366.000000
1,"Arun Jaitley Stadium, Delhi",364.312500
15,"Eden Gardens, Kolkata",362.250000


- Venues such as Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium,M Chinnaswamy Stadium  consistently produce high average scores.
- These venues likely favor batting due to pitch conditions, shorter boundaries, or ground dimensions.

#### Stage 6: Reporting

In [71]:
runs_per_match = runs_per_match.rename(columns={
    'match_id': 'match_id',
    'total_runs': 'total_runs'
}).sort_values('total_runs', ascending=False)

In [72]:
runs_per_team_match = runs_per_team_match.rename(columns={
    'batting_team': 'team',
    'team_runs': 'total_runs'
}).sort_values(['match_id', 'total_runs'], ascending=[True, False])

In [73]:
top_batters = top_batters.rename(columns={
    'batter': 'batter',
    'total_runs': 'total_runs'
}).sort_values('total_runs', ascending=False)

In [74]:
batter_stats = batter_stats.rename(columns={
    'batter': 'batter',
    'runs': 'total_runs',
    'balls': 'balls_faced',
    'strike_rate': 'strike_rate'
}).sort_values('strike_rate', ascending=False)

In [76]:
top_economy_bowlers = top_economy_bowlers.rename(columns={
    'bowler': 'bowler',
    'economy': 'economy'
}).sort_values('economy')

In [77]:
consistent_batters = consistent_batters.rename(columns={
    'batter': 'batter',
    'matches': 'matches_played',
    'avg_runs': 'avg_runs_per_match'
}).sort_values('avg_runs_per_match', ascending=False)

In [78]:
top_boundary_batter = top_boundary_batter.rename(columns={
    'batter': 'batter',
    'boundary_count': 'boundary_count'
}).sort_values('boundary_count', ascending=False)

In [79]:
boundary_percentage = boundary_percentage.rename(columns={
    'batter': 'batter',
    'boundary_percentage': 'boundary_percentage'
}).sort_values('boundary_percentage', ascending=False)

In [80]:
top_dot_bowlers = top_dot_bowlers.rename(columns={
    'bowler': 'bowler',
    'dot_balls': 'dot_balls'
}).sort_values('dot_balls', ascending=False)